# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abd481/Search-Ranks-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule will prioritize pages that have meaningful search visibility but may need a content refresh. A page will receive a higher score when it has enough impressions and shows signs that it may be stale or declining. The goal is to rank pages for human review, not to predict whether a refresh will definitely succeed.
Reason codes:
stale_visible_page: the page has meaningful search impressions and has not been updated recently.
declining_visible_page: the page has meaningful search impressions and shows a declining trend.
This is a simple baseline rule that uses observable signals only. The Week-5 ML model should try to improve on this ranking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse' 




In [4]:
con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
LIMIT 5
""").show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

In [5]:
baseline_query = """
WITH daily AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY content_hash_id
),

content AS (
    SELECT
        content_hash_id,
        content_created_date,
        content_updated_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    WHERE is_published IS TRUE
      AND is_deleted IS FALSE
)

SELECT
    d.content_hash_id,
    d.impressions,
    d.clicks,
    c.content_created_date,
    c.content_updated_date,

    DATE '2026-03-31'
        - COALESCE(
            CASE
                WHEN c.content_updated_date <= DATE '2026-03-31'
                THEN c.content_updated_date
            END,
            c.content_created_date
        ) AS days_since_update

FROM daily d
JOIN content c
    ON d.content_hash_id = c.content_hash_id
WHERE d.impressions > 0
"""

baseline_df = con.sql(baseline_query).df()

baseline_df.head()

,content_hash_id,impressions,clicks,content_created_date,content_updated_date,days_since_update
0,content_0263d5f9b7a2ecd4,1.0,0.0,2025-10-07,2026-05-20,175
1,content_04c67f3541177192,331.0,2.0,2025-10-07,2026-02-25,34
2,content_05acc92c165f4386,33.0,0.0,2025-10-07,2026-02-25,34
3,content_0f30e04e709c7b5d,145.0,0.0,2025-10-07,2026-02-25,34
4,content_1207efddce873942,461.0,0.0,2025-10-07,2026-05-20,175


In [6]:
import numpy as np

# Normalize the two signals to a 0-1 range.
# Higher values mean higher review priority.

baseline_df["visibility_score"] = (
    np.log1p(baseline_df["impressions"])
    / np.log1p(baseline_df["impressions"].max())
)

baseline_df["staleness_score"] = (
    baseline_df["days_since_update"].clip(lower=0) / 180
).clip(upper=1)

# Simple baseline:
# 60% visibility + 40% staleness
baseline_df["baseline_score"] = (
    0.60 * baseline_df["visibility_score"]
    + 0.40 * baseline_df["staleness_score"]
)

# One reason code, as required by the assignment.
baseline_df["reason_code"] = np.where(
    (baseline_df["days_since_update"] >= 180) &
    (baseline_df["impressions"] >= 100),
    "stale_visible_page",
    "review_candidate"
)

# Action label
baseline_df["action"] = "review_refresh"

# Rank highest-priority pages first
baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["rank"] = np.arange(1, len(baseline_df) + 1)

baseline_df.head(10)

,content_hash_id,impressions,clicks,content_created_date,content_updated_date,days_since_update,visibility_score,staleness_score,baseline_score,reason_code,action,rank
0,content_eadb33b5df496f4a,617124.0,5668.0,2025-03-21,2026-06-12,375,1.000000,1.0,1.000000,stale_visible_page,review_refresh,1
1,content_ec2e0346994fb5a5,245276.0,1480.0,2025-01-21,2026-06-12,434,0.930796,1.0,0.958478,stale_visible_page,review_refresh,2
2,content_e8a52cf3d5988c07,244931.0,669.0,2025-08-13,2026-06-11,230,0.930691,1.0,0.958414,stale_visible_page,review_refresh,3
3,content_0e03de7680314cd5,221310.0,720.0,2025-03-21,2026-06-12,375,0.923084,1.0,0.953851,stale_visible_page,review_refresh,4
4,content_e7b5dd4dff461ad2,205045.0,2446.0,2025-04-22,2026-06-11,343,0.917359,1.0,0.950415,stale_visible_page,review_refresh,5
5,content_8d7d99f109e19aa2,203497.0,289.0,2025-03-21,2026-06-12,375,0.916791,1.0,0.950074,stale_visible_page,review_refresh,6
6,content_36e53e9c707674fc,194579.0,242.0,2025-08-14,2026-06-11,229,0.913430,1.0,0.948058,stale_visible_page,review_refresh,7
7,content_4ffe18112a5642e3,186983.0,586.0,2025-03-21,2026-06-12,375,0.910443,1.0,0.946266,stale_visible_page,review_refresh,8
8,content_471d9cabce329a66,164885.0,396.0,2025-03-21,2026-06-26,375,0.901010,1.0,0.940606,stale_visible_page,review_refresh,9
9,content_512dbad65bd5ade9,154358.0,2506.0,2025-09-25,2026-06-26,187,0.896062,1.0,0.937637,stale_visible_page,review_refresh,10


In [7]:
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

baseline_df[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions",
        "days_since_update",
    ]
].to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(baseline_df):,}")

Saved: work/outputs/baseline_action_score.csv
Rows: 176,568


In [8]:
import pandas as pd 
pd.read_csv(output_path).head(10)

,rank,content_hash_id,baseline_score,reason_code,action,impressions,days_since_update
0,1,content_eadb33b5df496f4a,1.000000,stale_visible_page,review_refresh,617124.0,375
1,2,content_ec2e0346994fb5a5,0.958478,stale_visible_page,review_refresh,245276.0,434
2,3,content_e8a52cf3d5988c07,0.958414,stale_visible_page,review_refresh,244931.0,230
3,4,content_0e03de7680314cd5,0.953851,stale_visible_page,review_refresh,221310.0,375
4,5,content_e7b5dd4dff461ad2,0.950415,stale_visible_page,review_refresh,205045.0,343
5,6,content_8d7d99f109e19aa2,0.950074,stale_visible_page,review_refresh,203497.0,375
6,7,content_36e53e9c707674fc,0.948058,stale_visible_page,review_refresh,194579.0,229
7,8,content_4ffe18112a5642e3,0.946266,stale_visible_page,review_refresh,186983.0,375
8,9,content_471d9cabce329a66,0.940606,stale_visible_page,review_refresh,164885.0,375
9,10,content_512dbad65bd5ade9,0.937637,stale_visible_page,review_refresh,154358.0,187


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 pages produced by the baseline rule.

For each page, I record:
- the recommended action
- the reason code that triggered the recommendation
- a confidence note based on the observed signals
- what could make the recommendation wrong

The confidence notes are qualitative and do not represent probabilities. The review is used to identify obvious weak picks and limitations in the baseline rule.

### Top-20 review

| Rank | Action | Reason | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | review_refresh | stale_visible_page | High — very high visibility (617,124 impressions) and 375 days since update. | The page may still be accurate and intentionally not need a refresh. |
| 2 | review_refresh | stale_visible_page | High — strong visibility (245,276 impressions) and 434 days since update. | The content may still be accurate despite being old. |
| 3 | review_refresh | stale_visible_page | High — strong visibility (244,931 impressions) and 230 days since update. | The page may not have a meaningful refresh opportunity. |
| 4 | review_refresh | stale_visible_page | High — strong visibility (221,310 impressions) and 375 days since update. | The content may still be accurate and useful without changes. |
| 5 | review_refresh | stale_visible_page | High — strong visibility (205,045 impressions) and 343 days since update. | The page may be intentionally stable. |
| 6 | review_refresh | stale_visible_page | High — strong visibility (203,497 impressions) and 375 days since update. | The page may not need a content update. |
| 7 | review_refresh | stale_visible_page | High — strong visibility (194,579 impressions) and 229 days since update. | The page may still be current and accurate. |
| 8 | review_refresh | stale_visible_page | High — strong visibility (186,983 impressions) and 375 days since update. | The page may not have a useful refresh opportunity. |
| 9 | review_refresh | stale_visible_page | High — strong visibility (164,885 impressions) and 375 days since update. | The page may still be accurate despite its age. |
| 10 | review_refresh | stale_visible_page | High — strong visibility (154,358 impressions) and 187 days since update. | The page may be too recent to justify a refresh. |
| 11 | review_refresh | stale_visible_page | High — strong visibility (151,166 impressions) and 410 days since update. | The content may still be accurate and stable. |
| 12 | review_refresh | stale_visible_page | High — strong visibility (143,019 impressions) and 278 days since update. | The page may not require an update. |
| 13 | review_refresh | stale_visible_page | High — strong visibility (142,304 impressions) and 412 days since update. | The page may be intentionally stable. |
| 14 | review_refresh | stale_visible_page | High — strong visibility (140,156 impressions) and 230 days since update. | The page may still be current. |
| 15 | review_refresh | stale_visible_page | High — strong visibility (134,984 impressions) and 410 days since update. | The page may not have a meaningful refresh opportunity. |
| 16 | review_refresh | stale_visible_page | High — strong visibility (131,707 impressions) and 237 days since update. | The page may still be accurate without changes. |
| 17 | review_refresh | stale_visible_page | High — strong visibility (126,836 impressions) and 410 days since update. | The page may be intentionally stable. |
| 18 | review_refresh | stale_visible_page | High — strong visibility (124,720 impressions) and 259 days since update. | The page may still be current and useful. |
| 19 | review_refresh | stale_visible_page | High — strong visibility (124,075 impressions) and 410 days since update. | The page may not need a refresh despite its age. |
| 20 | review_refresh | stale_visible_page | High — strong visibility (122,905 impressions) and 375 days since update. | The content may still be accurate and intentionally unchanged. |

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = (
    baseline_df
    .sort_values("rank")
    .head(20)
    [["rank",
      "content_hash_id",
      "action",
      "reason_code",
      "baseline_score",
      "impressions",
      "days_since_update"]]
)

top20


,rank,content_hash_id,action,reason_code,baseline_score,impressions,days_since_update
0,1,content_eadb33b5df496f4a,review_refresh,stale_visible_page,1.000000,617124.0,375
1,2,content_ec2e0346994fb5a5,review_refresh,stale_visible_page,0.958478,245276.0,434
2,3,content_e8a52cf3d5988c07,review_refresh,stale_visible_page,0.958414,244931.0,230
3,4,content_0e03de7680314cd5,review_refresh,stale_visible_page,0.953851,221310.0,375
4,5,content_e7b5dd4dff461ad2,review_refresh,stale_visible_page,0.950415,205045.0,343
5,6,content_8d7d99f109e19aa2,review_refresh,stale_visible_page,0.950074,203497.0,375
6,7,content_36e53e9c707674fc,review_refresh,stale_visible_page,0.948058,194579.0,229
7,8,content_4ffe18112a5642e3,review_refresh,stale_visible_page,0.946266,186983.0,375
8,9,content_471d9cabce329a66,review_refresh,stale_visible_page,0.940606,164885.0,375
9,10,content_512dbad65bd5ade9,review_refresh,stale_visible_page,0.937637,154358.0,187


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline rule consistently selects pages with high visibility and high staleness as the highest-priority refresh candidates.

A potential weak pick is any page where age alone does not imply that the content needs updating. For example, a page that is intentionally stable, still accurate, or does not have a meaningful refresh opportunity could be incorrectly prioritized.

The baseline also has a limitation because the top 20 all share the same reason code. This means the rule may be too narrow and does not distinguish between different types of refresh opportunities.

### Leakage check

The baseline score uses only observable inputs available at the decision point, such as impressions and content age/staleness.

I did not use product flags, future-window metrics, or label-derived fields to calculate the score. Product flags are specifically treated as leakage-risk context in the FlyRank data guidance. citeturn0search0turn0search1

The recommendation is therefore a prioritization rule, not a prediction that a refresh will cause future traffic growth.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the inputs used by the baseline rule
baseline_df[
    [
        "impressions",
        "days_since_update",
        "visibility_score",
        "staleness_score",
        "baseline_score",
        "reason_code",
        "action",
    ]
].head() 

print("Top 20 rows:", len(baseline_df.head(20)))
print("Reason codes:", baseline_df.head(20)["reason_code"].value_counts().to_dict())
print("Actions:", baseline_df.head(20)["action"].value_counts().to_dict()) 

Top 20 rows: 20
Reason codes: {'stale_visible_page': 20}
Actions: {'review_refresh': 20}


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.